# 05. Feature Extraction

> Compute features known to be informative for motor decoding from ECoG: band power in mu (8–12 Hz), beta (13–30 Hz), low-gamma (30–70 Hz), and high-gamma (70–170 Hz) ranges.

Also covers time-frequency representations and Common Spatial Pattern (CSP) filters as alternatives. Each function takes an epoched `(trials × channels × time)` tensor and returns a `(trials × features)` matrix ready for the classifiers in `06_classification`.

In [ ]:
#| default_exp features

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from scipy.signal import welch

## Frequency bands

Standard motor-decoding bands. High-gamma is the workhorse for ECoG hand decoding, but the mu/beta event-related desynchronization (ERD) is informative too.

In [ ]:
#| export
BANDS = {
    'mu':         (8, 12),
    'beta':       (13, 30),
    'low_gamma':  (30, 70),
    'high_gamma': (70, 170),
}

## Single-band power

Average PSD power between `low` and `high` Hz. Returns `(trials, channels)` when given an epoched tensor, or `(channels,)` when given continuous data.

In [ ]:
#| export
def band_power(x, fs, low, high, nperseg=None):
    """Mean PSD power in `[low, high]` Hz along the last axis of `x`."""
    n = x.shape[-1] if nperseg is None else min(nperseg, x.shape[-1])
    freqs, psd = welch(x, fs=fs, nperseg=n, axis=-1)
    mask = (freqs >= low) & (freqs <= high)
    return psd[..., mask].mean(axis=-1)

## Multi-band feature matrix

Stack band power across `bands` into a `(trials, channels × n_bands)` matrix ready for a classifier. Log-transform is on by default because band power is approximately log-normal — the transformed features fit Gaussian classifiers (LDA, QDA) much better.

In [ ]:
#| export
def multi_band_power(epochs, fs, bands=None, log=True):
    """Concatenate band-power features over `bands` along the last axis."""
    bands = BANDS if bands is None else bands
    feats = []
    for low, high in bands.values():
        bp = band_power(epochs, fs, low, high)
        feats.append(np.log(bp + 1e-12) if log else bp)
    return np.concatenate(feats, axis=-1)

## Z-score normalization

Center each feature and scale to unit variance. Fit on the training fold; apply to the test fold. Avoids per-channel amplitude differences dominating the classifier.

In [ ]:
#| export
def zscore_fit(X):
    """Return `(mean, std)` along axis 0 for later application via `zscore_apply`."""
    return X.mean(axis=0), X.std(axis=0) + 1e-12

def zscore_apply(X, stats):
    mean, std = stats
    return (X - mean) / std

## Visual validation

Plot mean log-band-power per class for the high-gamma band. If the classes look separable here, a linear classifier should do well in `06`.

In [ ]:
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
from br41n_ecog_hand_pose.data import load_ecog, GESTURE_NAMES
from br41n_ecog_hand_pose.preprocessing import preprocess
from br41n_ecog_hand_pose.epoching import epoch_recording

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

In [ ]:
#| eval: false
rec = load_ecog()
clean, _ = preprocess(rec.ecog, rec.fs)
epochs, classes = epoch_recording(rec, tmin=0.0, tmax=2.0, signal=clean)
X = multi_band_power(epochs, rec.fs)
print(f'epochs:   {epochs.shape}')
print(f'features: {X.shape}     # 60 channels \u00d7 {len(__import__("br41n_ecog_hand_pose.features", fromlist=["BANDS"]).BANDS)} bands')

In [ ]:
#| eval: false
# split features back into (trials, channels, bands) for plotting
n_ch = epochs.shape[1]
F = X.reshape(X.shape[0], len(BANDS), n_ch).transpose(0, 2, 1)  # (trials, channels, bands)

fig, axs = plt.subplots(1, len(BANDS), figsize=(4 * len(BANDS), 3), sharey=True)
for ax, (band_name, _), bi in zip(axs, BANDS.items(), range(len(BANDS))):
    for c in [1, 2, 3]:
        ax.plot(F[classes == c, :, bi].mean(axis=0), label=GESTURE_NAMES[c])
    ax.set_title(band_name)
    ax.set_xlabel('channel')
axs[0].set_ylabel('mean log-power')
axs[0].legend(fontsize=8)
fig.suptitle('Mean log-band-power per channel, per class')
plt.tight_layout(); plt.show()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()